# 06 — Live Forecast

End-to-end forecast pipeline in a single cell:

```
ticker + horizon → prices → regimes → Markov signal
                    ↓
              Polymarket markets → PM signal
                    ↓
              Ensemble blend → forecast price + CI + quality
                    ↓
              Action signal: BUY / HOLD / SELL
```

**Prerequisites:**
```bash
conda activate cramer-research
cd ../research && pip install -e . && cd ../notebooks
```

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from research.data.prices import fetch_historical_prices
from research.data.polymarket import fetch_polymarket_markets
from research.models.markov import (
    classify_regime_series,
    estimate_transition_matrix,
    compute_markov_forecast,
)
from research.models.ensemble import (
    MarketInput,
    OtherSignals,
    run_ensemble,
)

## 1. Live Forecast Function

Single function that ties all modules together.

In [ ]:
def compute_regime_up_rates(regimes, returns, horizon, decay_rate=0.97):
    counts = {s: {'up': 0.0, 'total': 0.0} for s in ['bull', 'bear', 'sideways']}
    max_start = min(len(regimes), len(returns)) - horizon
    for i in range(max_start):
        regime = regimes[i]
        cum_return = sum(returns[i+1:i+1+horizon])
        weight = decay_rate ** (max_start - 1 - i)
        counts[regime]['total'] += weight
        if cum_return > 0:
            counts[regime]['up'] += weight
    return {
        s: counts[s]['up'] / counts[s]['total'] if counts[s]['total'] > 0 else 0.5
        for s in ['bull', 'bear', 'sideways']
    }


def run_live_forecast(ticker, horizon=7, current_price=None):
    """End-to-end forecast for a single ticker.
    
    Returns a dict with all forecast components.
    """
    # Fetch prices
    prices = fetch_historical_prices(ticker, days=120)
    cp = current_price if current_price is not None else float(prices['close'].iloc[-1])
    returns = prices['close'].pct_change().dropna().values
    
    # Markov signal
    regimes = classify_regime_series(returns)
    P = estimate_transition_matrix(regimes, decay_rate=0.97)
    forecast = compute_markov_forecast(P, regimes[-1], horizon)
    up = compute_regime_up_rates(regimes, returns, horizon)
    p_up = sum(forecast[s] * up[s] for s in ['bull', 'bear', 'sideways'])
    markov_return = p_up - 0.5
    
    # Polymarket markets
    query = ticker.lower().replace('-usd', '').replace('usd', '')
    markets_df = fetch_polymarket_markets(query, limit=10)
    
    market_inputs = []
    for _, row in markets_df.iterrows():
        market_inputs.append(MarketInput(
            question=row['question'],
            probability=row['probability'],
            volume24h_usd=row['volume_24h'],
            age_days=int(row['age_days']) if pd.notna(row['age_days']) else None,
            signal_tier='macro',
            delta_yes=0.06,
            delta_no=-0.04,
        ))
    
    # Auxiliary signals (placeholder values)
    others = OtherSignals(
        sentiment_score=0.3,
        fundamental_return=0.02,
        options_skew=0.5,
        markov_return=markov_return,
        horizon_days=horizon,
    )
    
    # Ensemble
    result = run_ensemble(current_price=cp, markets=market_inputs, others=others)
    
    # Action signal
    expected_return = result.forecast_return
    risk = result.sigma
    rr = expected_return / risk if risk > 0 else 0
    
    if expected_return > 0.02 and rr > 1.0:
        action = 'BUY'
    elif expected_return < -0.02 and rr < -1.0:
        action = 'SELL'
    else:
        action = 'HOLD'
    
    target = cp * (1 + expected_return + result.sigma)
    stop = cp * (1 + expected_return - result.sigma * 2)
    
    return {
        'ticker': ticker,
        'horizon': horizon,
        'current_price': cp,
        'forecast_price': result.forecast_price,
        'forecast_return': expected_return,
        'ci_low': result.ci_low95,
        'ci_high': result.ci_high95,
        'sigma': result.sigma,
        'quality_score': result.quality_score,
        'quality_grade': result.quality_grade,
        'action': action,
        'risk_reward': rr,
        'target': target,
        'stop': stop,
        'pm_weight': result.pm_effective_weight,
        'markov_return': markov_return,
        'pm_markets': len(market_inputs),
        'warnings': result.warnings,
        'weights': result.weights,
    }

# Test on BTC
result = run_live_forecast('BTC', horizon=7)
print(f"=== {result['ticker']} {result['horizon']}-Day Forecast ===" )
for k, v in result.items():
    if k not in ('warnings', 'weights'):
        print(f"  {k:20s}: {v}")

## 2. Multi-Ticker Comparison

Run the pipeline for multiple tickers side-by-side.

In [ ]:
TICKERS = ['BTC', 'ETH', 'SOL']
HORIZON = 7

results = [run_live_forecast(t, HORIZON) for t in TICKERS]

summary = pd.DataFrame([
    {
        'Ticker': r['ticker'],
        'Current': f"${r['current_price']:,.0f}",
        'Forecast': f"${r['forecast_price']:,.0f}",
        'Return': f"{r['forecast_return']*100:+.2f}%",
        '95% CI': f"[${r['ci_low']:,.0f}, ${r['ci_high']:,.0f}]",
        'Quality': f"{r['quality_score']}/100 ({r['quality_grade']})",
        'Action': r['action'],
        'R/R': f"{r['risk_reward']:.2f}",
        'PM Mkts': r['pm_markets'],
    }
    for r in results
])

print(summary.to_string(index=False))

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, r in zip(axes, results):
    cp = r['current_price']
    fp = r['forecast_price']
    low = r['ci_low']
    high = r['ci_high']
    
    ax.bar(['Current', 'Forecast'], [cp, fp], color=['gray', 'steelblue'])
    ax.errorbar([1], [fp], yerr=[[fp - low], [high - fp]], fmt='o', color='red', capsize=10)
    ax.set_title(f"{r['ticker']}: {r['action']}")
    ax.set_ylabel('Price')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'{HORIZON}-Day Forecast Comparison', y=1.02)
plt.tight_layout()
plt.show()

## 3. Risk Metrics

Compute risk-adjusted metrics for each forecast.

In [ ]:
def compute_risk_metrics(ticker, days=120):
    """Compute volatility, max drawdown, and Sharpe-like ratio."""
    prices = fetch_historical_prices(ticker, days=days)
    returns = prices['close'].pct_change().dropna()
    
    vol = returns.std() * np.sqrt(252)  # annualised
    mean_ret = returns.mean() * 252
    sharpe = mean_ret / vol if vol > 0 else 0
    
    # Max drawdown
    cum = (1 + returns).cumprod()
    peak = cum.expanding().max()
    dd = (cum - peak) / peak
    max_dd = dd.min()
    
    return {
        'ticker': ticker,
        'annual_return': mean_ret,
        'annual_vol': vol,
        'sharpe': sharpe,
        'max_drawdown': max_dd,
    }

risk_data = [compute_risk_metrics(t) for t in TICKERS]
risk_df = pd.DataFrame(risk_data)
print(risk_df.to_string(index=False))

## 4. Horizon Sweep

How does the forecast change across horizons?

In [ ]:
TICKER = 'BTC'
HORIZONS = [1, 3, 7, 14, 30]

sweep_results = []
for h in HORIZONS:
    r = run_live_forecast(TICKER, horizon=h)
    sweep_results.append({
        'horizon': h,
        'return_pct': r['forecast_return'] * 100,
        'ci_low': r['ci_low'],
        'ci_high': r['ci_high'],
        'quality': r['quality_score'],
        'action': r['action'],
    })

sweep_df = pd.DataFrame(sweep_results)

fig, ax = plt.subplots(figsize=(12, 6))
ax.errorbar(
    sweep_df['horizon'],
    sweep_df['return_pct'],
    yerr=[sweep_df['return_pct'] - (sweep_df['ci_low'] / result['current_price'] * 100 - 100),
          (sweep_df['ci_high'] / result['current_price'] * 100 - 100) - sweep_df['return_pct']],
    fmt='o',
    markersize=10,
    capsize=8,
    linewidth=2,
    label='Forecast Return ± CI',
)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Horizon (days)')
ax.set_ylabel('Forecast Return (%)')
ax.set_title(f"{TICKER}: Forecast Across Horizons")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(sweep_df.to_string(index=False))

## 5. Report Generator

Generate a clean text report suitable for copy-paste.

In [ ]:
def generate_report(ticker, horizon=7):
    r = run_live_forecast(ticker, horizon=horizon)
    lines = [
        f"═══════════════════════════════════════════════════════",
        f"  {ticker} {horizon}-Day Forecast Report",
        f"═══════════════════════════════════════════════════════",
        f"",
        f"  Current Price:     ${r['current_price']:,.2f}",
        f"  Forecast Price:    ${r['forecast_price']:,.2f}  ({r['forecast_return']*100:+.2f}%)",
        f"  95% CI:            [${r['ci_low']:,.2f}, ${r['ci_high']:,.2f}]",
        f"  Sigma:             {r['sigma']:.4f}",
        f"",
        f"  Quality Score:     {r['quality_score']}/100  (Grade {r['quality_grade']})",
        f"  PM Markets Used:   {r['pm_markets']}",
        f"  PM Weight:         {r['pm_weight']*100:.1f}%",
        f"",
        f"  ─── Action Signal ───",
        f"  Signal:            {r['action']}",
        f"  Target:            ${r['target']:,.2f}",
        f"  Stop:              ${r['stop']:,.2f}",
        f"  Risk/Reward:       {r['risk_reward']:.2f}",
        f"",
        f"  ─── Signal Weights ───",
    ]
    for name, w in r['weights'].items():
        lines.append(f"  {name:15s}: {w*100:5.1f}%")
    if r['warnings']:
        lines.append(f"")
        lines.append(f"  ─── Warnings ───")
        for w in r['warnings'][:5]:
            lines.append(f"  • {w}")
    lines.append(f"")
    lines.append(f"═══════════════════════════════════════════════════════")
    return "\n".join(lines)

print(generate_report('BTC', horizon=7))